# GDP-Driven Equity Allocation

Varies SPY exposure using unannualized quarter-over-quarter growth in nominal US GDP. The strategy observes each signal at the session close and executes at the following session’s open, including the configured commission and slippage.

The default uses the latest revised [FRED GDP series](https://fred.stlouisfed.org/series/GDP) and assigns each quarter an assumed availability date 45 calendar days after quarter end. This is an approximation with revision hindsight. For a release-aware run, set `GDP_VINTAGE_CSV` to a local CSV with one row per quarter and the columns `quarter_end,publication_date,GDP`. Use ISO dates and the nominal level known on each listed release date, in consistent units. Supply earlier quarters to calculate the first growth signal. The [BEA GDP archive](https://www.bea.gov/data/gdp/gross-domestic-product) and [ALFRED observations API](https://fred.stlouisfed.org/docs/api/fred/series_observations.html) provide historical vintages.

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import os
from io import StringIO

from _market_data import download_ohlcv
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION PARAMETERS - All inputs are configurable here
# ============================================================================

# Date Range Configuration
START_DATE = '2019-01-01'  # 5 years of data as requested
END_DATE = '2024-12-31'
SPLIT_DATE = '2022-01-01'  # IS/OOS split date

# Trading Asset Configuration
TRADING_TICKER = 'SPY'  # Asset to trade dynamically
BENCHMARK_TICKER = 'SPY'  # Benchmark

# GDP Signal Configuration
GDP_PUBLICATION_LAG_DAYS = 45  # Assumed lag after quarter end for revised FRED GDP
GDP_VINTAGE_CSV = os.environ.get("GDP_VINTAGE_CSV")
GDP_GROWTH_WINDOW = 1  # Number of quarters to calculate growth (1 = QoQ growth)

# Weight Calculation Parameters - Dynamic allocation based on GDP growth
# These thresholds determine portfolio allocation to risky asset
WEIGHT_PARAMS = {
    'min_weight': 0.20,      # Minimum allocation (defensive mode)
    'neutral_weight': 0.50,  # Neutral allocation
    'max_weight': 1.00,      # Maximum allocation (aggressive mode)
    'gdp_threshold_low': -2.0,    # GDP growth below this → min_weight
    'gdp_threshold_neutral': 0.0,  # GDP growth at this → neutral_weight
    'gdp_threshold_mid': 2.0,      # GDP growth at this → 70% weight
    'gdp_threshold_high': 4.0      # GDP growth above this → max_weight
}

# Transaction Costs Configuration
COMMISSION_PCT = 0.001  # 0.1% commission per trade (realistic for institutional)
SLIPPAGE_PCT = 0.0005   # 0.05% slippage per trade

# Capital Configuration
INITIAL_CAPITAL = 100000  # Starting capital in USD

# ============================================================================
# Read quarterly GDP and map the date it becomes observable in the backtest.

def download_gdp_data(vintage_csv):
    if vintage_csv:
        data = pd.read_csv(vintage_csv, usecols=["quarter_end", "publication_date", "GDP"])
        data["quarter_end"] = pd.to_datetime(data["quarter_end"], format="%Y-%m-%d")
        data["Publication_Date"] = pd.to_datetime(data.pop("publication_date"), format="%Y-%m-%d")
        if data[["quarter_end", "Publication_Date"]].isna().any().any():
            raise ValueError("Each GDP vintage row needs a quarter-end and publication date.")
        if not data["quarter_end"].dt.is_quarter_end.all():
            raise ValueError("Use calendar quarter ends in quarter_end.")
        if (data["Publication_Date"] <= data["quarter_end"]).any():
            raise ValueError("GDP publication dates must follow the associated quarter end.")
        data = data.set_index("quarter_end")
    else:
        response = requests.get(
            "https://fred.stlouisfed.org/graph/fredgraph.csv", params={"id": "GDP"}, timeout=30
        )
        response.raise_for_status()
        data = pd.read_csv(StringIO(response.text))
        date_column = data.columns[0]
        dates = pd.to_datetime(data.pop(date_column), format="%Y-%m-%d")
        data.index = dates.dt.to_period("Q").dt.end_time.dt.normalize()
    data["GDP"] = pd.to_numeric(data["GDP"], errors="coerce")
    data = data.dropna(subset=["GDP"]).sort_index()
    if data.empty or not data.index.is_unique or (data["GDP"] <= 0).any():
        raise ValueError("Supply positive nominal GDP with exactly one observation per quarter.")
    return data


def process_gdp_signals(gdp_data, publication_lag_days, growth_window, trading_calendar):
    signals = gdp_data.copy()
    signals["GDP_Growth"] = signals["GDP"].pct_change(periods=growth_window, fill_method=None) * 100
    if "Publication_Date" not in signals:
        signals["Publication_Date"] = signals.index + pd.Timedelta(days=publication_lag_days)
    signals = signals.dropna(subset=["GDP_Growth"])
    signals = signals[
        signals["Publication_Date"].between(trading_calendar[0], trading_calendar[-1])
    ].sort_values("Publication_Date")
    next_sessions = trading_calendar.searchsorted(signals["Publication_Date"])
    signals["Observation_Trading_Day"] = trading_calendar[next_sessions]
    signals = signals.drop_duplicates("Observation_Trading_Day", keep="last")
    if signals.empty:
        raise ValueError("No GDP observations became available inside the trading period.")
    return signals.set_index("Observation_Trading_Day")


def calculate_dynamic_weight(gdp_growth, params):
    """
    Calculate portfolio weight in risky asset based on GDP growth.

    ALLOCATION LOGIC:
    - Strong GDP growth (>4%) → 100% in risky asset (aggressive)
    - Moderate growth (2-4%) → 70-100% allocation
    - Weak growth (0-2%) → 50-70% allocation (neutral)
    - Contraction (-2-0%) → 30-50% allocation
    - Deep contraction (<-2%) → 20% allocation (defensive)

    Remainder held in cash (0% return, 0% volatility)

    Parameters:
    -----------
    gdp_growth : float
        Quarterly GDP growth rate in percentage
    params : dict
        Weight calculation parameters (thresholds and weights)

    Returns:
    --------
    float : Weight in risky asset (0.0 to 1.0)
    """
    # Handle NaN values (no GDP data yet)
    if pd.isna(gdp_growth):
        return params['neutral_weight']

    # Extract thresholds and weights
    t_low = params['gdp_threshold_low']
    t_neutral = params['gdp_threshold_neutral']
    t_mid = params['gdp_threshold_mid']
    t_high = params['gdp_threshold_high']

    w_min = params['min_weight']
    w_neutral = params['neutral_weight']
    w_max = params['max_weight']

    # Piecewise linear mapping
    if gdp_growth < t_low:
        # Deep contraction: minimum weight
        return w_min
    elif gdp_growth < t_neutral:
        # Contraction to zero: interpolate from min to neutral
        slope = (w_neutral - w_min) / (t_neutral - t_low)
        return w_min + slope * (gdp_growth - t_low)
    elif gdp_growth < t_mid:
        # Weak positive growth: interpolate from neutral to 70%
        w_mid = 0.70
        slope = (w_mid - w_neutral) / (t_mid - t_neutral)
        return w_neutral + slope * (gdp_growth - t_neutral)
    elif gdp_growth < t_high:
        # Strong growth: interpolate from 70% to max
        w_mid = 0.70
        slope = (w_max - w_mid) / (t_high - t_mid)
        return w_mid + slope * (gdp_growth - t_mid)
    else:
        # Very strong growth: maximum weight
        return w_max

# ============================================================================
# SECTION 4: BACKTEST ENGINE WITH CORRECTED REBALANCE VALUATION
# ============================================================================

def run_backtest(asset_data, benchmark_data, gdp_signals, params,
                 commission_pct, slippage_pct, initial_capital):
    """
    Run backtest with quarterly rebalancing on GDP publication dates.

    CORRECTED TRADING FLOW:
    ========================

    GDP Observation Day T (trading day, could be Mon if pub was on weekend):
    - GDP data available at market close
    - Signal at Close[T]: We observe GDP, calculate target weight for tomorrow

    Execution Day T+1:
    - CRITICAL FIX: Value existing position at Open[T+1] (not Close[T])
    - This captures the overnight gap and gives true portfolio value at execution
    - Execute rebalance at Open[T+1] based on actual available capital
    - Buy/sell shares to reach target weight

    Holding Period (T+1 to next GDP release, ~90 days):
    - Hold position constant
    - Calculate returns Close-to-Close for ALL days
    - No intraday data used

    RETURNS CALCULATION:
    - ALL days: Return = Close[T] / Close[T-1] - 1
    - Consistent Close-to-Close measurement
    - Benchmark also uses Close-to-Close (fair comparison)

    VALUATION FIX EXAMPLE:
    - Close[May 15]: $100, holding 100 shares + $0 cash
    - Signal: Reduce weight from 100% to 50%
    - Open[May 16]: $102 (gap up +2%)
    - WRONG: Value at Close[May 15] = 100 * $100 = $10,000
    - CORRECT: Value at Open[May 16] = 100 * $102 = $10,200
    - Target position = $10,200 * 50% = $5,100 (not $5,000)
    - The overnight gap affects how much capital we actually have to deploy

    Parameters:
    -----------
    asset_data : pd.DataFrame
        OHLC data for trading asset
    benchmark_data : pd.DataFrame
        OHLC data for benchmark
    gdp_signals : pd.DataFrame
        GDP data with observation trading day as index
    params : dict
        Weight calculation parameters
    commission_pct : float
        Commission percentage per trade
    slippage_pct : float
        Slippage percentage per trade
    initial_capital : float
        Starting capital

    Returns:
    --------
    pd.DataFrame with backtest results
    """
    print("\nRunning backtest with quarterly rebalancing...")

    # Prepare daily price data
    df = asset_data[['Open', 'Close']].copy()
    df.columns = ['Asset_Open', 'Asset_Close']
    df['Benchmark_Close'] = benchmark_data['Close']

    # Calculate daily Close-to-Close returns for the asset and benchmark
    # These are the returns we'll earn during holding periods
    df['Asset_Daily_Return'] = df['Asset_Close'].pct_change()
    df['Benchmark_Daily_Return'] = df['Benchmark_Close'].pct_change()

    # Initialize columns for tracking
    df['Target_Weight'] = np.nan  # Target weight from GDP signal
    df['Position_Weight'] = 0.0   # Actual weight held
    df['Cash'] = float(initial_capital)
    df['Portfolio_Value'] = float(initial_capital)
    df['Is_Rebalance_Day'] = False  # Flag for rebalance execution days
    df['Trade_Cost'] = 0.0
    df['GDP_Growth'] = np.nan
    df['Shares_Held'] = 0.0

    # ========================================================================
    # STEP 1: Mark GDP observation dates and set target weights
    # ========================================================================
    # On each GDP observation date (trading day after publication),
    # we observe the new GDP at close and decide target weight for tomorrow

    gdp_observation_days = []
    for obs_date in gdp_signals.index:
        if obs_date in df.index:
            gdp_growth = gdp_signals.loc[obs_date, 'GDP_Growth']
            target_weight = calculate_dynamic_weight(gdp_growth, params)

            df.loc[obs_date, 'Target_Weight'] = target_weight
            df.loc[obs_date, 'GDP_Growth'] = gdp_growth
            gdp_observation_days.append(obs_date)

    print(f"  GDP observation days mapped: {len(gdp_observation_days)}")

    # Forward-fill target weights (we maintain same weight until next GDP)
    df['Target_Weight'] = df['Target_Weight'].ffill()
    df['GDP_Growth'] = df['GDP_Growth'].ffill()

    # Fill initial NaN with neutral weight (before first GDP observation)
    df['Target_Weight'] = df['Target_Weight'].fillna(params['neutral_weight'])

    # ========================================================================
    # STEP 2: Identify rebalance execution days (day after GDP observation)
    # ========================================================================
    # We execute rebalances the trading day AFTER GDP observation
    # This follows "decide at close, trade next open" convention

    for obs_date in gdp_observation_days:
        obs_idx = df.index.get_loc(obs_date)
        # Mark next trading day as rebalance day
        if obs_idx + 1 < len(df):
            exec_date = df.index[obs_idx + 1]
            df.loc[exec_date, 'Is_Rebalance_Day'] = True

    # Also rebalance on first day (establish initial position)
    df.iloc[0, df.columns.get_loc('Is_Rebalance_Day')] = True

    rebalance_count = df['Is_Rebalance_Day'].sum()
    print(f"  Rebalance execution days: {rebalance_count}")

    # ========================================================================
    # STEP 3: Execute backtest day by day
    # ========================================================================

    current_shares = 0.0
    current_cash = float(initial_capital)

    for i in range(len(df)):
        date = df.index[i]
        is_rebalance = df.iloc[i]['Is_Rebalance_Day']
        target_weight = df.iloc[i]['Target_Weight']
        asset_open = df.iloc[i]['Asset_Open']
        asset_close = df.iloc[i]['Asset_Close']

        # ====================================================================
        # A) REBALANCE DAY: Execute at Open
        # ====================================================================
        if is_rebalance:
            # CRITICAL FIX: Value existing position at TODAY'S OPEN
            # This captures the overnight gap between yesterday's close and today's open
            # We need to know the actual portfolio value at the moment of execution
            existing_position_value = current_shares * asset_open
            total_value_at_open = current_cash + existing_position_value

            # Calculate target position value
            target_position_value = total_value_at_open * target_weight

            # Calculate trade size and costs
            trade_value = abs(target_position_value - existing_position_value)
            trade_cost = trade_value * (commission_pct + slippage_pct)

            # Execute trade at Open
            # New shares to hold after rebalancing
            new_shares = target_position_value / asset_open if asset_open > 0 else 0

            # Cash after trade: total value - position value - costs
            new_cash = total_value_at_open - target_position_value - trade_cost

            # Update state
            current_shares = new_shares
            current_cash = new_cash

            df.iloc[i, df.columns.get_loc('Trade_Cost')] = trade_cost
            df.iloc[i, df.columns.get_loc('Shares_Held')] = current_shares
        else:
            # Non-rebalance day: carry forward position
            df.iloc[i, df.columns.get_loc('Shares_Held')] = current_shares

        # ====================================================================
        # B) VALUE PORTFOLIO AT CLOSE (all days)
        # ====================================================================
        # Standard practice: Value portfolio at Close for consistent measurement
        # This applies to both rebalance days and holding days

        position_value_at_close = current_shares * asset_close
        portfolio_value_at_close = position_value_at_close + current_cash

        df.iloc[i, df.columns.get_loc('Position_Weight')] = (
            position_value_at_close / portfolio_value_at_close
            if portfolio_value_at_close > 0 else 0
        )
        df.iloc[i, df.columns.get_loc('Portfolio_Value')] = portfolio_value_at_close
        df.iloc[i, df.columns.get_loc('Cash')] = current_cash

    # ========================================================================
    # STEP 4: Calculate portfolio returns (Close-to-Close)
    # ========================================================================
    # ALL returns calculated Close-to-Close for consistency
    # This matches benchmark returns (also Close-to-Close)
    df['Strategy_Return'] = df['Portfolio_Value'].pct_change()

    # Fill first return as 0
    df.iloc[0, df.columns.get_loc('Strategy_Return')] = 0.0

    # Remove any NaN
    df['Strategy_Return'] = df['Strategy_Return'].fillna(0)
    df['Benchmark_Daily_Return'] = df['Benchmark_Daily_Return'].fillna(0)

    print(f"\n✓ Backtest completed:")
    print(f"  - Total days: {len(df)}")
    print(f"  - Total rebalances: {rebalance_count}")
    print(f"  - Total transaction costs: ${df['Trade_Cost'].sum():.2f}")
    print(f"  - Final portfolio value: ${df['Portfolio_Value'].iloc[-1]:,.2f}")
    print(f"  - Final position: {df['Shares_Held'].iloc[-1]:.2f} shares @ ${df.iloc[-1]['Asset_Close']:.2f}")

    return df

# ============================================================================
# SECTION 5: PERFORMANCE METRICS CALCULATION
# ============================================================================

def calculate_metrics(data, label):
    """
    Calculate comprehensive performance metrics.

    All metrics use Close-to-Close returns consistently.

    Metrics calculated:
    - Total and annualized returns
    - Volatility (annualized)
    - Sharpe ratio (assuming rf=0)
    - Maximum drawdown
    - Number of rebalances
    - Total costs

    Parameters:
    -----------
    data : pd.DataFrame
        Backtest results
    label : str
        Label for this period (e.g., "In-Sample")

    Returns:
    --------
    dict : Performance metrics
    """
    print(f"\nCalculating metrics for {label}...")

    # Cumulative returns
    strategy_cumret = (1 + data['Strategy_Return']).cumprod()
    benchmark_cumret = (1 + data['Benchmark_Daily_Return']).cumprod()

    # Time period
    total_days = len(data)
    years = total_days / 252.0

    # Total returns
    strategy_total = (strategy_cumret.iloc[-1] - 1) * 100
    benchmark_total = (benchmark_cumret.iloc[-1] - 1) * 100

    # Annualized returns
    strategy_annual = ((strategy_cumret.iloc[-1]) ** (1/years) - 1) * 100
    benchmark_annual = ((benchmark_cumret.iloc[-1]) ** (1/years) - 1) * 100

    # Volatility
    strategy_vol = data['Strategy_Return'].std() * np.sqrt(252) * 100
    benchmark_vol = data['Benchmark_Daily_Return'].std() * np.sqrt(252) * 100

    # Sharpe ratio
    strategy_sharpe = strategy_annual / strategy_vol if strategy_vol > 0 else 0
    benchmark_sharpe = benchmark_annual / benchmark_vol if benchmark_vol > 0 else 0

    # Maximum drawdown
    strategy_dd = ((strategy_cumret / strategy_cumret.cummax()) - 1).min() * 100
    benchmark_dd = ((benchmark_cumret / benchmark_cumret.cummax()) - 1).min() * 100

    # Trade statistics
    num_rebalances = data['Is_Rebalance_Day'].sum()
    total_costs = data['Trade_Cost'].sum()
    avg_weight = data['Position_Weight'].mean() * 100

    metrics = {
        'cumret': (strategy_cumret, benchmark_cumret),
        'metrics': {
            'GDP Strategy': {
                'Total Return': f"{strategy_total:.2f}%",
                'Annual Return': f"{strategy_annual:.2f}%",
                'Volatility': f"{strategy_vol:.2f}%",
                'Sharpe Ratio': f"{strategy_sharpe:.2f}",
                'Max Drawdown': f"{strategy_dd:.2f}%",
                'Avg Weight': f"{avg_weight:.1f}%",
                'Rebalances': f"{int(num_rebalances)}",
                'Total Costs': f"${total_costs:.2f}"
            },
            f'Buy & Hold ({BENCHMARK_TICKER})': {
                'Total Return': f"{benchmark_total:.2f}%",
                'Annual Return': f"{benchmark_annual:.2f}%",
                'Volatility': f"{benchmark_vol:.2f}%",
                'Sharpe Ratio': f"{benchmark_sharpe:.2f}",
                'Max Drawdown': f"{benchmark_dd:.2f}%",
                'Avg Weight': '100.0%',
                'Rebalances': '1',
                'Total Costs': '$0.00'
            }
        }
    }

    return metrics

# ============================================================================
# SECTION 6: VISUALIZATION
# ============================================================================

def create_visualizations(is_data, oos_data, is_metrics, oos_metrics):
    """
    Create comprehensive visualization of backtest results.

    Parameters:
    -----------
    is_data : pd.DataFrame
        In-sample backtest results
    oos_data : pd.DataFrame
        Out-of-sample backtest results
    is_metrics : dict
        In-sample performance metrics
    oos_metrics : dict
        Out-of-sample performance metrics
    """
    print("\nGenerating visualizations...")

    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle(f'GDP Dynamic Allocation Strategy: {TRADING_TICKER} vs {BENCHMARK_TICKER}',
                 fontsize=16, fontweight='bold')

    # IS: Cumulative returns
    ax1 = axes[0, 0]
    is_strat, is_bench = is_metrics['cumret']
    ax1.plot(is_strat.index, (is_strat - 1) * 100, label='GDP Strategy', linewidth=2)
    ax1.plot(is_bench.index, (is_bench - 1) * 100, label=f'{BENCHMARK_TICKER}',
             linewidth=2, alpha=0.7)
    # Mark rebalance days
    rebal_days = is_data[is_data['Is_Rebalance_Day']].index
    for day in rebal_days:
        if day in is_strat.index:
            ax1.axvline(x=day, color='gray', alpha=0.2, linewidth=0.5, linestyle='--')
    ax1.set_title('In-Sample: Cumulative Returns (Close-to-Close)', fontweight='bold', fontsize=12)
    ax1.set_ylabel('Return (%)', fontsize=10)
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # OOS: Cumulative returns
    ax2 = axes[0, 1]
    oos_strat, oos_bench = oos_metrics['cumret']
    ax2.plot(oos_strat.index, (oos_strat - 1) * 100, label='GDP Strategy', linewidth=2)
    ax2.plot(oos_bench.index, (oos_bench - 1) * 100, label=f'{BENCHMARK_TICKER}',
             linewidth=2, alpha=0.7)
    # Mark rebalance days
    rebal_days = oos_data[oos_data['Is_Rebalance_Day']].index
    for day in rebal_days:
        if day in oos_strat.index:
            ax2.axvline(x=day, color='gray', alpha=0.2, linewidth=0.5, linestyle='--')
    ax2.set_title('Out-of-Sample: Cumulative Returns (Close-to-Close)', fontweight='bold', fontsize=12)
    ax2.set_ylabel('Return (%)', fontsize=10)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    # IS: GDP Growth with rebalance markers
    ax3 = axes[1, 0]
    ax3.plot(is_data.index, is_data['GDP_Growth'], color='green', linewidth=1.5, label='GDP Growth')
    ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax3.fill_between(is_data.index, 0, is_data['GDP_Growth'],
                     where=is_data['GDP_Growth']>=0, alpha=0.3, color='green')
    ax3.fill_between(is_data.index, 0, is_data['GDP_Growth'],
                     where=is_data['GDP_Growth']<0, alpha=0.3, color='red')
    # Mark rebalance days
    rebal_days = is_data[is_data['Is_Rebalance_Day']].index
    for day in rebal_days:
        ax3.axvline(x=day, color='blue', alpha=0.3, linewidth=1, linestyle=':')
    ax3.set_title('In-Sample: GDP Growth (QoQ %) - Blue lines = Rebalances',
                  fontweight='bold', fontsize=12)
    ax3.set_ylabel('Growth Rate (%)', fontsize=10)
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)

    # OOS: GDP Growth with rebalance markers
    ax4 = axes[1, 1]
    ax4.plot(oos_data.index, oos_data['GDP_Growth'], color='green', linewidth=1.5, label='GDP Growth')
    ax4.axhline(y=0, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax4.fill_between(oos_data.index, 0, oos_data['GDP_Growth'],
                     where=oos_data['GDP_Growth']>=0, alpha=0.3, color='green')
    ax4.fill_between(oos_data.index, 0, oos_data['GDP_Growth'],
                     where=oos_data['GDP_Growth']<0, alpha=0.3, color='red')
    # Mark rebalance days
    rebal_days = oos_data[oos_data['Is_Rebalance_Day']].index
    for day in rebal_days:
        ax4.axvline(x=day, color='blue', alpha=0.3, linewidth=1, linestyle=':')
    ax4.set_title('Out-of-Sample: GDP Growth (QoQ %) - Blue lines = Rebalances',
                  fontweight='bold', fontsize=12)
    ax4.set_ylabel('Growth Rate (%)', fontsize=10)
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3)

    # IS: Dynamic weights
    ax5 = axes[2, 0]
    ax5.plot(is_data.index, is_data['Position_Weight'] * 100,
             color='purple', linewidth=1.5, label='Actual Weight')
    ax5.fill_between(is_data.index, 0, is_data['Position_Weight'] * 100,
                     alpha=0.3, color='purple')
    # Mark rebalance days
    rebal_days = is_data[is_data['Is_Rebalance_Day']].index
    for day in rebal_days:
        ax5.axvline(x=day, color='blue', alpha=0.3, linewidth=1, linestyle=':')
    ax5.set_title(f'In-Sample: Dynamic Allocation to {TRADING_TICKER}',
                  fontweight='bold', fontsize=12)
    ax5.set_ylabel('Weight (%)', fontsize=10)
    ax5.set_ylim([0, 105])
    ax5.axhline(y=WEIGHT_PARAMS['neutral_weight']*100, color='orange',
                linestyle='--', alpha=0.5, linewidth=1, label='Neutral')
    ax5.legend(fontsize=9)
    ax5.grid(True, alpha=0.3)

    # OOS: Dynamic weights
    ax6 = axes[2, 1]
    ax6.plot(oos_data.index, oos_data['Position_Weight'] * 100,
             color='purple', linewidth=1.5, label='Actual Weight')
    ax6.fill_between(oos_data.index, 0, oos_data['Position_Weight'] * 100,
                     alpha=0.3, color='purple')
    # Mark rebalance days
    rebal_days = oos_data[oos_data['Is_Rebalance_Day']].index
    for day in rebal_days:
        ax6.axvline(x=day, color='blue', alpha=0.3, linewidth=1, linestyle=':')
    ax6.set_title(f'Out-of-Sample: Dynamic Allocation to {TRADING_TICKER}',
                  fontweight='bold', fontsize=12)
    ax6.set_ylabel('Weight (%)', fontsize=10)
    ax6.set_ylim([0, 105])
    ax6.axhline(y=WEIGHT_PARAMS['neutral_weight']*100, color='orange',
                linestyle='--', alpha=0.5, linewidth=1, label='Neutral')
    ax6.legend(fontsize=9)
    ax6.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function.
    """
    print("=" * 80)
    print("GDP-DRIVEN EQUITY ALLOCATION")
    print("=" * 80)
    print(f"\nConfiguration:")
    print(f"  Trading Asset: {TRADING_TICKER}")
    print(f"  Benchmark: {BENCHMARK_TICKER}")
    print(f"  Period: {START_DATE} to {END_DATE}")
    print(f"  IS/OOS Split: {SPLIT_DATE}")
    print(f"  Commission: {COMMISSION_PCT*100:.2f}%")
    print(f"  Slippage: {SLIPPAGE_PCT*100:.3f}%")
    if GDP_VINTAGE_CSV:
        print("  GDP source: vintage CSV with supplied publication dates")
    else:
        print(f"  GDP source: revised FRED data with assumed {GDP_PUBLICATION_LAG_DAYS}-day lag")
    print(f"  Rebalancing: Quarterly (only on GDP publication)")
    print(f"  Returns: Close-to-Close (consistent horizon)")
    print(f"  Valuation on rebalance: At Open (captures overnight gap)")

    # Download data
    print("\n" + "=" * 80)
    print("STEP 1: DATA DOWNLOAD")
    print("=" * 80)

    asset_data = download_ohlcv(TRADING_TICKER, START_DATE, END_DATE)
    benchmark_data = asset_data if BENCHMARK_TICKER == TRADING_TICKER else download_ohlcv(
        BENCHMARK_TICKER, START_DATE, END_DATE
    )
    gdp_data = download_gdp_data(GDP_VINTAGE_CSV)

    # Get trading calendar from asset data
    trading_calendar = asset_data.index

    # Process GDP signals
    print("\n" + "=" * 80)
    print("STEP 2: SIGNAL PROCESSING")
    print("=" * 80)

    gdp_signals = process_gdp_signals(gdp_data, GDP_PUBLICATION_LAG_DAYS,
                                     GDP_GROWTH_WINDOW, trading_calendar)

    # Run backtest
    print("\n" + "=" * 80)
    print("STEP 3: BACKTEST EXECUTION")
    print("=" * 80)

    results = run_backtest(asset_data, benchmark_data, gdp_signals, WEIGHT_PARAMS,
                          COMMISSION_PCT, SLIPPAGE_PCT, INITIAL_CAPITAL)

    # Split IS/OOS
    print("\n" + "=" * 80)
    print("STEP 4: IN-SAMPLE / OUT-OF-SAMPLE ANALYSIS")
    print("=" * 80)

    is_data = results[results.index < SPLIT_DATE].copy()
    oos_data = results[results.index >= SPLIT_DATE].copy()

    print(f"\nIn-Sample Period: {is_data.index[0].date()} to {is_data.index[-1].date()}")
    print(f"  Days: {len(is_data)}")
    print(f"  Rebalances: {is_data['Is_Rebalance_Day'].sum()}")

    print(f"\nOut-of-Sample Period: {oos_data.index[0].date()} to {oos_data.index[-1].date()}")
    print(f"  Days: {len(oos_data)}")
    print(f"  Rebalances: {oos_data['Is_Rebalance_Day'].sum()}")

    # Calculate metrics
    is_metrics = calculate_metrics(is_data, "In-Sample")
    oos_metrics = calculate_metrics(oos_data, "Out-of-Sample")

    # Display results
    print("\n" + "=" * 80)
    print("IN-SAMPLE RESULTS")
    print("=" * 80)
    for strategy, metrics in is_metrics['metrics'].items():
        print(f"\n{strategy}:")
        for metric, value in metrics.items():
            print(f"  {metric:.<30} {value:>15}")

    print("\n" + "=" * 80)
    print("OUT-OF-SAMPLE RESULTS")
    print("=" * 80)
    for strategy, metrics in oos_metrics['metrics'].items():
        print(f"\n{strategy}:")
        for metric, value in metrics.items():
            print(f"  {metric:.<30} {value:>15}")

    # Visualizations
    print("\n" + "=" * 80)
    print("STEP 5: VISUALIZATION")
    print("=" * 80)

    create_visualizations(is_data, oos_data, is_metrics, oos_metrics)

    # Final summary


    print("\n" + "=" * 80)
    print("BACKTEST COMPLETED SUCCESSFULLY")
    print("=" * 80)

    return results, is_metrics, oos_metrics

# ============================================================================
# RUN THE STRATEGY
# ============================================================================

if __name__ == "__main__":
    results, is_metrics, oos_metrics = main()
